# Ensemble learning I: bagging and random forests

**Lecture 15 · Notebook 00 · CMOR 438 / INDE 577**

One deep decision tree can change dramatically when only a few observations
change. This laboratory asks a precise question:

> **Can we convert the instability of trees from a weakness into useful
> diversity, then average that diversity into a more reliable predictor?**

We will derive the averaging argument, simulate it, implement the mechanics,
and compare a single tree, bagging, random forests, and extremely randomized
trees on the real Wisconsin Diagnostic Breast Cancer data. The goal is not to
declare one universal winner. It is to understand *why* an ensemble may win and
what evidence a professional model review must still demand.

## How to use this notebook

Run every cell in order using the **Rice DSM** kernel. Read the mathematics
before running its experiment, predict the qualitative result, and then explain
any disagreement. The `rice_dsm.ml.ensembles` module contains durable code;
this notebook is an experimental laboratory and evidence narrative.

**Route:** population and data contract → instability → bootstrap → variance
of an average → bagging → random forests → out-of-bag evidence → interpretation
and probability diagnostics → contribution boundary.

## Learning objectives

By the end you should be able to:

1. distinguish an ensemble, bagging, a random forest, and extra trees;
2. derive how member-error correlation limits variance reduction;
3. explain bootstrap sampling and the approximately 63.2% unique-row fact;
4. interpret out-of-bag predictions without confusing them with final test data;
5. compare discrimination, probability quality, latency, and stability; and
6. place reusable forest code and tests in the companion student project.

## Why this matters: a short historical and scientific view

The idea that a group can outperform one judgment predates machine learning.
Modern ensemble learning made the mechanism algorithmic. Leo Breiman introduced
**bagging** (bootstrap aggregating) in 1996 and **random forests** in 2001.
Tin Kam Ho's random-subspace work and extremely randomized trees provide closely
related routes to diversity.

The key is not “many models are always better.” Members must be competent, and
their errors must not be identical. Randomization deliberately perturbs the
training problem. Averaging then suppresses variation that is not shared.

**Professional practice.** Record the data version, split rule, seed, estimator
configuration, class threshold, fit time, prediction time, and model size. A
forest is a fitted artifact with an operating contract—not merely an accuracy
number.

In [ ]:
# Import the data-access and plotting tools shared by this laboratory.
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from rice_dsm.paths import course_database_path

RANDOM_STATE = 438
DATABASE_PATH = course_database_path()
assert DATABASE_PATH.is_file(), (
    "Course database missing. From the repository root run: "
    "uv run --locked python scripts/refresh_course.py"
)
sns.set_theme(style="whitegrid", context="notebook")
print("Database:", DATABASE_PATH)
print("Random seed:", RANDOM_STATE)

## 1. Why average? The correlation calculation

Let each prediction error be $E_m=F_m(X)-Y$ with
$\operatorname{Var}(E_m)=\sigma^2$. Suppose every distinct pair has
$\operatorname{Corr}(E_m,E_\ell)=\rho$. For the average error
$\bar E=M^{-1}\sum_{m=1}^{M}E_m$,

\[
\begin{aligned}
\operatorname{Var}(\bar E)
&=\frac{1}{M^2}\left(\sum_m\operatorname{Var}(E_m)
+\sum_{m\ne \ell}\operatorname{Cov}(E_m,E_\ell)\right)\\
&=\frac{1}{M^2}\left(M\sigma^2+M(M-1)\rho\sigma^2\right)\\
&=\rho\sigma^2+\frac{1-\rho}{M}\sigma^2.
\end{aligned}
\]

The $1/M$ term vanishes as the ensemble grows; the shared-error floor
$\rho\sigma^2$ does not. This is the mathematical reason random forests try to
reduce correlation without destroying individual tree quality.

In [ ]:
# Simulate correlated errors using one shared shock plus independent shocks.
ensemble_sizes = np.arange(1, 201)
correlations = [0.0, 0.1, 0.4, 0.8, 1.0]
fig, ax = plt.subplots(figsize=(9, 5))
for rho in correlations:
    theoretical_variance = rho + (1.0 - rho) / ensemble_sizes
    ax.plot(ensemble_sizes, theoretical_variance, label=fr"$\rho={rho}$")
ax.set(xlabel="number of members M", ylabel="variance of average / member variance",
       title="Averaging cannot remove shared error")
ax.legend(ncol=3)
plt.show()

assert np.isclose(correlations[0] + (1-correlations[0])/100, 0.01)
assert np.isclose(correlations[-1] + (1-correlations[-1])/100, 1.0)

## 2. Bootstrap sampling: perturb rows while preserving size

Given $n$ observations, draw $n$ indices **with replacement**. The probability
that a particular observation is omitted is $(1-1/n)^n\to e^{-1}\approx0.368$.
Thus a large bootstrap sample contains about $1-e^{-1}\approx63.2\%$ unique
training rows. Omitted rows are **out of bag (OOB)** for that member.

Bagging fits one unstable learner per bootstrap sample. Classification usually
aggregates votes or probabilities; regression averages numerical predictions.
The bootstrap does not create new information. It creates related perturbations
of the empirical training distribution.

In [ ]:
from rice_dsm.ml.ensembles import bootstrap_indices

# Inspect one tiny draw: repeated indices are in-bag; omitted indices are OOB.
draw = bootstrap_indices(12, random_state=RANDOM_STATE)
unique_rows = np.unique(draw)
oob_rows = np.setdiff1d(np.arange(12), unique_rows)
print("bootstrap indices:", draw)
print("unique in-bag rows:", unique_rows)
print("out-of-bag rows:", oob_rows)

assert draw.size == 12
assert unique_rows.size < draw.size
assert np.intersect1d(unique_rows, oob_rows).size == 0

## 3. Worked example: a real diagnostic dataset

Each row is a digitized fine-needle-aspirate image summarized by measured cell
nucleus features. The target distinguishes benign from malignant cases. This is
a teaching benchmark, **not** a clinical device: the repository data do not
represent a current deployment population, intervention, workflow, or cost
study.

Before modeling we inspect schema, units, class support, ranges, and correlations.
Why? A model can run successfully while answering the wrong question, leaking a
target surrogate, or relying on an implausible range.

In [ ]:
# Query the versioned SQLite table in read-only mode.
database_uri = f"file:{DATABASE_PATH.as_posix()}?mode=ro"
with sqlite3.connect(database_uri, uri=True) as connection:
    cancer = pd.read_sql_query(
        "SELECT * FROM breast_cancer_observations ORDER BY observation_id",
        connection,
    )

feature_names = [
    "mean_radius", "mean_texture", "mean_perimeter", "mean_area",
    "mean_smoothness", "mean_compactness", "mean_concavity",
    "mean_concave_points", "worst_radius", "worst_texture",
]
target_name = "diagnosis_code"
# Inspect rows, distributions, uniqueness, and missingness before any fit.
display(cancer[["observation_id", *feature_names, "diagnosis_label"]].head())
display(cancer[feature_names].describe().T)
print(cancer["diagnosis_label"].value_counts())

assert cancer.shape[0] == 569
assert cancer[feature_names + [target_name]].isna().sum().sum() == 0
assert cancer["observation_id"].is_unique
assert set(cancer[target_name]) == {0, 1}

In [ ]:
# Visualize relationships before fitting; the target is used only for color.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(data=cancer, x="mean_radius", y="mean_texture",
                hue="diagnosis_label", alpha=0.75, ax=axes[0])
correlations = cancer[feature_names].corr()
sns.heatmap(correlations, cmap="vlag", center=0, ax=axes[1], cbar_kws={"shrink": .7})
axes[0].set_title("Two measurements do not perfectly separate diagnoses")
axes[1].set_title("Many predictors are strongly redundant")
plt.tight_layout()
plt.show()

assert correlations.shape == (len(feature_names), len(feature_names))
assert np.allclose(np.diag(correlations), 1.0)

### Freeze the information boundary

The untouched test set represents the final audit. Hyperparameter comparisons
use only training data through cross-validation. Stratification preserves class
support approximately; it does not guarantee transport to another hospital,
time period, or acquisition system.

In [ ]:
from sklearn.model_selection import train_test_split

# Freeze one stratified test audit before comparing any models.
X = cancer[feature_names].to_numpy()
y = cancer[target_name].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

print("train:", X_train.shape, "test:", X_test.shape)
print("malignant fraction:", y_train.mean(), y_test.mean())
assert set(map(tuple, X_train)).isdisjoint(set(map(tuple, X_test)))
assert abs(y_train.mean() - y_test.mean()) < 0.03

## 4. Bagging versus a random forest

**Bagging:** bootstrap rows; grow each tree using all candidate features at a
split; average.

**Random forest:** bootstrap rows **and** expose each split to only a random
subset of features. If one strong feature dominates, feature subsampling stops
every tree from making the same first decision. This may slightly weaken each
tree while lowering error correlation enough to improve the average.

**Extra trees:** additionally randomize candidate thresholds, often reducing
variance and training cost at the price of possible extra bias.

Standardization is generally unnecessary for axis-aligned trees: an order-
preserving rescaling does not change which observations lie on either side of a
threshold. Missing-data policy, categories, sample weights, and leakage still
require explicit handling.

In [ ]:
from sklearn.ensemble import BaggingClassifier, ExtraTreesClassifier
from sklearn.metrics import balanced_accuracy_score, log_loss, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from rice_dsm.ml.ensembles import RandomForestClassifier

# Keep configurations readable: this is a mechanism comparison, not a search contest.
models = {
    "one tree": DecisionTreeClassifier(min_samples_leaf=3, random_state=RANDOM_STATE),
    "bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(min_samples_leaf=3),
        n_estimators=60, random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "course forest": RandomForestClassifier(
        n_estimators=60, max_features="sqrt", min_samples_leaf=3,
        random_state=RANDOM_STATE,
    ),
    "extra trees": ExtraTreesClassifier(
        n_estimators=60, max_features="sqrt", min_samples_leaf=3,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

# Use identical held-out rows and metrics so the model comparison is meaningful.
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_test)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    rows.append({
        "model": name,
        "balanced_accuracy": balanced_accuracy_score(y_test, prediction),
        "ROC_AUC": roc_auc_score(y_test, probability),
        "log_loss": log_loss(y_test, probability),
    })
results = pd.DataFrame(rows).set_index("model").sort_values("log_loss")
display(results.round(3))

assert results.shape == (4, 3)
assert np.isfinite(results.to_numpy()).all()

## 5. Inspect the course forest rather than treating it as magic

The course implementation delegates individual split optimization to a trusted
CART tree, while explicitly owning bootstrap draws, seed generation, probability
alignment, averaging, and OOB accumulation. That division illustrates a useful
engineering principle: write the mechanism you need to understand or control;
reuse a verified component whose contract you can test.

In [ ]:
# Inspect stored samples and member scores to verify the aggregation mechanism.
forest = models["course forest"]
first_draw = forest.bootstrap_indices_[0]
unique_fraction = np.unique(first_draw).size / first_draw.size
member_probabilities = np.vstack([
    tree.predict_proba(X_test)[:, 1] for tree in forest.estimators_
])

print(f"unique fraction in first bootstrap sample: {unique_fraction:.3f}")
print(f"OOB coverage: {(forest.oob_counts_ > 0).mean():.3f}")
print(f"OOB accuracy: {forest.oob_score_:.3f}")
print("member probability matrix:", member_probabilities.shape)

assert 0.5 < unique_fraction < 0.75
assert np.all(forest.oob_counts_ > 0)
np.testing.assert_allclose(
    forest.predict_proba(X_test)[:, 1], member_probabilities.mean(axis=0)
)

In [ ]:
# Measure prediction diversity through pairwise correlation of member scores.
correlation_matrix = np.corrcoef(member_probabilities)
upper_triangle = correlation_matrix[np.triu_indices_from(correlation_matrix, k=1)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(upper_triangle, bins=20, ax=axes[0])
axes[0].set(xlabel="pairwise correlation", title="Forest members remain related")

# Show stabilization: the cumulative mean should move less as trees accumulate.
cumulative = np.cumsum(member_probabilities, axis=0) / np.arange(1, 61)[:, None]
axes[1].plot(np.arange(1, 61), cumulative[:, :12], alpha=0.55)
axes[1].set(xlabel="number of trees", ylabel="mean malignant probability",
            title="Selected predictions stabilize with ensemble size")
plt.tight_layout()
plt.show()

assert -1 <= upper_triangle.mean() <= 1
assert cumulative.shape == member_probabilities.shape

## 6. OOB evidence: useful, internal, and limited

For each training row, aggregate predictions only from trees whose bootstrap
samples omitted that row. This gives nearly free internal evidence and is useful
for debugging and rough model selection. It is **not** an excuse to inspect the
test set repeatedly, and it does not repair the wrong dependence structure. If
patients, sites, or time batches repeat, bootstrap and validation units must
respect those groups.

In [ ]:
# Compare internal OOB evidence with the separately held-out test audit.
from sklearn.metrics import ConfusionMatrixDisplay

oob_probability = forest.oob_decision_function_[:, 1]
oob_prediction = (oob_probability >= 0.5).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_train, oob_prediction, ax=axes[0], cmap="Blues")
axes[0].set_title("Out-of-bag training evidence")
ConfusionMatrixDisplay.from_predictions(y_test, forest.predict(X_test), ax=axes[1], cmap="Oranges")
axes[1].set_title("Untouched test audit")
plt.tight_layout()
plt.show()

assert oob_probability.shape == y_train.shape
assert ((0 <= oob_probability) & (oob_probability <= 1)).all()

## 7. Interpretation: impurity and permutation importance are not scientific truth

Mean decrease in impurity is fast but favors features with many candidate split
points and divides credit unpredictably among correlated predictors. Permutation
importance asks how much held-out performance degrades when a column is
scrambled. It is closer to an operational question, but correlated substitutes
can hide one another. Neither establishes causality.

In [ ]:
from sklearn.inspection import permutation_importance

permutation = permutation_importance(
    forest, X_test, y_test, scoring="balanced_accuracy",
    n_repeats=20, random_state=RANDOM_STATE,
)
# Put both importance definitions on one table before plotting their disagreement.
importance = pd.DataFrame({
    "feature": feature_names,
    "impurity": forest.feature_importances_,
    "permutation_mean": permutation.importances_mean,
    "permutation_sd": permutation.importances_std,
}).sort_values("permutation_mean")

# Error bars show variation across repeated held-out permutations.
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
importance.plot.barh(x="feature", y="impurity", legend=False, ax=axes[0])
importance.plot.barh(x="feature", y="permutation_mean", xerr="permutation_sd",
                     legend=False, ax=axes[1])
axes[0].set_title("Mean impurity decrease")
axes[1].set_title("Held-out permutation importance")
plt.tight_layout()
plt.show()

assert importance.shape[0] == len(feature_names)
assert np.isclose(forest.feature_importances_.sum(), 1.0)

## 8. Probability quality, thresholds, and resource evidence

ROC AUC evaluates ranking, not whether 0.8 means an event occurs about 80% of
the time. Log loss penalizes confident errors; calibration curves compare
predicted and empirical frequencies. Threshold choice belongs to the decision
problem: false negatives and false positives can have different consequences.

A champion–challenger report also measures fit time, batch and tail latency,
serialized size, memory, determinism, subgroup behavior, and drift sensitivity.
More trees often stabilize a forest, but they still increase resource cost.

In [ ]:
# Calibration curves ask whether probabilities match observed event frequencies.
from sklearn.calibration import CalibrationDisplay

fig, ax = plt.subplots(figsize=(6.5, 5))
for name in ["one tree", "course forest", "extra trees"]:
    probability = models[name].predict_proba(X_test)[:, 1]
    CalibrationDisplay.from_predictions(y_test, probability, n_bins=8, name=name, ax=ax)
ax.set_title("Probability evidence is distinct from ranking")
plt.tight_layout()
plt.show()

assert len(ax.lines) >= 3

## Common failure modes and debugging

- **Every tree is identical:** verify bootstrap draws, feature randomness, and seeds.
- **Suspiciously perfect OOB score:** check whether the target or a post-outcome
  field entered the features and whether grouped entities cross boundaries.
- **Probabilities do not sum to one:** align member class orders before averaging.
- **Results change across runs:** control every randomized component, environment,
  and data query ordering.
- **Forest is slow:** profile before reducing evidence; use parallel production
  implementations and measure memory/latency explicitly.
- **Importance becomes a causal claim:** restate it as model reliance under a
  specified perturbation and validate with domain knowledge.

## Guided practice

1. Change `max_features` among `None`, `"sqrt"`, and `0.5`. Record member-score
   correlation, OOB score, test log loss, and fit time.
2. Compare `min_samples_leaf=1, 3, 10`. Explain the probability-resolution and
   bias–variance tradeoff.
3. Choose a threshold under a stated false-negative/false-positive cost ratio.

Do not choose a setting from the final test result. Use training-only OOB or
cross-validation evidence, freeze the choice, then audit once.

In [ ]:
# PRACTICE: replace these candidate values, then explain the evidence.
candidate_max_features = "sqrt"
candidate_leaf_size = 3
practice_model = RandomForestClassifier(
    n_estimators=30, max_features=candidate_max_features,
    min_samples_leaf=candidate_leaf_size, random_state=RANDOM_STATE,
).fit(X_train, y_train)

assert practice_model.oob_score_ > 0.5
assert practice_model.predict_proba(X_test).shape == (len(X_test), 2)

## Independent exercise: student contribution test plan

In the **companion student template**, implement or wrap a forest for your own
scientific question.

1. Put reusable code under `src/dsm_project/algorithms/student/forests.py`.
2. Put a lawful, tiny CI fixture under `data/fixtures/`; keep full supplied data
   under ignored `data/local/` and record its source and license.
3. Add `tests/test_student_forests.py` covering reproducibility, probability
   shape and sums, prediction-before-fit, invalid hyperparameters, and one
   trusted-library differential comparison.
4. Import the package into a question-led notebook; do not reimplement it there.
5. Open a focused pull request containing the data contract, evidence table,
   limitations, and reviewer instructions.

In [ ]:
# Run these commands in the companion repository terminal, not this cell:
# git switch main
# git pull --ff-only
# git switch -c student/forest-<short-question>
# uv sync --locked
# uv run ruff check .
# uv run pytest -q
# git add src tests notebooks data/fixtures
# git commit -m "Add random-forest analysis for <question>"
# git push -u origin student/forest-<short-question>
assert True

## Extension

Investigate grouped or temporal resampling, class-weighted forests, quantile
forests, conformal prediction around forest outputs, or conditional permutation
importance. State which new claim the extension can support and which assumption
it introduces.

## Retrieval practice

1. What two sources of randomness distinguish a random forest from one tree?
2. Why does averaging fail to eliminate highly correlated error?
3. Where does the 63.2% unique-row approximation come from?
4. Why is OOB evidence not automatically safe for repeated patients or time?
5. Why can two valid importance methods disagree?

## Takeaway

Bagging reduces variance by averaging unstable learners fitted to bootstrap
perturbations. Random forests also randomize candidate features to lower member
correlation. Their gains are empirical and conditional: preserve the information
boundary, audit probabilities and decisions, and measure operational cost.

## Further reading

- Breiman, “Bagging Predictors” (1996).
- Breiman, “Random Forests” (2001).
- scikit-learn [ensemble guide](https://scikit-learn.org/stable/modules/ensemble.html).
- Python [`statistics`](https://docs.python.org/3/library/statistics.html) documentation
  for the vocabulary of means and variance.